# Actividad 5A. Cuaderno Jupyter: Flujo completo de procesamiento y reconocimiento
**Procesamiento Digital de Imágenes · MCDI · INFOTEC · 2027-1** · Unidad 5

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/egonzalez1989/pdi-mcdi/blob/main/plantillas/5A_Plantilla_Flujo_completo.ipynb)

| | |
|---|---|
| **Estudiante** | *Escribe aquí tu nombre completo* |
| **Tipo de actividad** | Individual |
| **Fecha límite** | Domingo 16 de mayo de 2027, 23:59 hrs. |
| **Puntaje** | 30 puntos |

**Propósito.** Diseñar, implementar y evaluar un flujo completo de procesamiento de imágenes y aprendizaje automático para un problema de clasificación o agrupamiento.

**Producto esperado.** Una libreta Jupyter bien documentada que incluya:
- Definición del problema (clasificación o agrupamiento) y descripción del conjunto de datos.
- Preprocesamiento, segmentación (si es relevante) y extracción de descriptores.
- Matriz de características y etiquetas; entrenamiento y evaluación de al menos dos modelos o de un método no supervisado.
- Análisis de resultados, limitaciones y mejoras, documentado en Markdown.

### Instrucciones
1. Descarga el cuaderno plantilla 5A_Plantilla_Flujo_completo.ipynb. Define qué quieres distinguir o agrupar e indica cuántas imágenes utilizarás, de dónde provienen y cómo las organizaste (carpetas, nombres o tabla de etiquetas).
2. Preprocesamiento: aplica las operaciones necesarias (cambio de tamaño, conversión de color, filtrado, normalización) y justifica brevemente cada una.
3. Segmentación (opcional, recomendable si trabajas con objetos sobre fondo): aplica alguno de los métodos de la Unidad 3 y explica si ayuda a definir las regiones de interés.
4. Extracción de características: define los descriptores (histogramas, GLCM/LBP, momentos de Hu, color) y construye la matriz de características, verificando su dimensión.
5. Modelo supervisado: separa entrenamiento y prueba de forma estratificada (por ejemplo, 70 %–30 %, con semilla fija); construye un Pipeline (estandarización + clasificador) y compara al menos dos modelos (k-NN, SVM, árbol o Random Forest) con validación cruzada. Si varias imágenes provienen de la misma fuente, usa partición por grupos para evitar fuga de datos. Alternativa no supervisada: aplica k-means y analiza la relación de los grupos con tus etiquetas.
6. Evaluación: reporta matriz de confusión, exactitud y F1 (macro y por clase) en el conjunto de prueba, y muestra ejemplos de imágenes mal clasificadas. Opcional: compara tus descriptores con embeddings de una red preentrenada (ResNet-18), como se muestra en la sesión síncrona.
7. Análisis y discusión: ¿qué tan bien funcionó el flujo?, ¿cuáles fueron las principales fuentes de error?, ¿qué cambiarías en una versión mejorada (más datos, otros descriptores, otro modelo)?
8. Organiza tu libreta en secciones (Problema y datos, Preprocesamiento, Segmentación, Características, Modelo y resultados, Discusión) de modo que alguien que no tomó el curso entienda qué hiciste y por qué. Guárdala como 5A_Nombre_Apellido.ipynb, expórtala a PDF con el mismo nombre y entrega ambos archivos.

### Lista de cotejo (así se evaluará tu trabajo)
| Criterio | Puntaje |
|---|---|
| Definición del problema y descripción del conjunto de datos. | 6 |
| Diseño e implementación del flujo (preprocesamiento, segmentación opcional, extracción de características). | 8 |
| Implementación y evaluación del modelo de aprendizaje automático (Pipeline, validación cruzada, matriz de confusión y F1). | 8 |
| Claridad en el análisis de resultados, conclusiones y documentación. | 8 |
| **Total** | **30** |

> 💡 Escribe tus interpretaciones en celdas Markdown como esta. Una figura sin interpretación no demuestra aprendizaje. Si usas fragmentos de código de otras fuentes (incluidos asistentes de IA), cítalos en la sección final.

> Usa el cuaderno de la **sesión 6** (caso modelo) como referencia de estructura. La comparación con una red preentrenada de la **sesión 5** es opcional.

In [ ]:
# Configuración del entorno. En Google Colab todas estas bibliotecas ya vienen instaladas.
import sys, platform
import numpy as np
import matplotlib.pyplot as plt
import cv2
import skimage
from skimage import data

plt.rcParams.update({"figure.dpi": 110, "axes.titlesize": 10, "image.cmap": "gray"})
np.random.seed(42)

def mostrar(imagenes, titulos=None, cols=None, cmap="gray", tam=3.2, ejes=False):
    """Muestra una lista de imágenes en una cuadrícula. Las imágenes a color deben estar en RGB."""
    if not isinstance(imagenes, (list, tuple)):
        imagenes = [imagenes]
    n = len(imagenes)
    cols = cols or min(n, 4)
    filas = int(np.ceil(n / cols))
    fig, axs = plt.subplots(filas, cols, figsize=(tam * cols, tam * filas), squeeze=False)
    for i, ax in enumerate(axs.flat):
        if i < n:
            im = imagenes[i]
            ax.imshow(im, cmap=None if im.ndim == 3 else cmap)
            if titulos:
                ax.set_title(titulos[i])
        if not ejes or i >= n:
            ax.axis("off")
    plt.tight_layout()
    plt.show()

print(f"Python {platform.python_version()} | NumPy {np.__version__} | OpenCV {cv2.__version__} | scikit-image {skimage.__version__}")

# Datos del curso: se buscan en la carpeta datos/ del repositorio (si lo clonaste) o se descargan.
import os, io, zipfile, urllib.request
def carpeta_datos(url="https://github.com/egonzalez1989/pdi-mcdi/archive/refs/heads/main.zip"):
    for d in ("datos", "../datos", "pdi-mcdi-main/datos"):
        if os.path.isdir(d):
            return d
    try:
        with urllib.request.urlopen(url, timeout=20) as r:
            zipfile.ZipFile(io.BytesIO(r.read())).extractall(".")
        return "pdi-mcdi-main/datos"
    except Exception as e:
        print("No se pudieron descargar los datos del curso:", e)
        return None
DATOS = carpeta_datos()
print("Carpeta de datos:", DATOS)

import glob
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, f1_score
SEMILLA = 2027

## 1. Problema y datos
| Elemento | Descripción |
|---|---|
| ¿Qué quiero distinguir o agrupar? | ... |
| Origen de las imágenes | ... |
| Número de imágenes y clases | ... |
| Organización (carpetas / etiquetas) | ... |

In [ ]:
CARPETA = ""   # una subcarpeta por clase; por omisión, datos/texturas
if not CARPETA and DATOS:
    CARPETA = os.path.join(DATOS, "texturas")
df = pd.DataFrame([{"ruta": r, "clase": os.path.basename(os.path.dirname(r))}
                   for r in sorted(glob.glob(os.path.join(CARPETA, "*", "*")))
                   if r.lower().endswith((".png", ".jpg", ".jpeg", ".tif", ".bmp"))])
print(df.clase.value_counts() if not df.empty else "No se encontraron imágenes")

## 2. Preprocesamiento

In [ ]:
def preprocesar(img):
    """TODO: cambio de tamaño, conversión de color, filtrado, normalización... justifica cada paso."""
    return img

## 3. Segmentación (opcional)

In [ ]:
# TODO opcional: umbralización, Canny + morfología o k-means para aislar la región de interés

## 4. Extracción de características

In [ ]:
def caracteristicas(img):
    """TODO: devuelve un diccionario con tus descriptores (puedes reutilizar las funciones de la 4A)."""
    return {}

# X = pd.DataFrame([caracteristicas(preprocesar(cv2.imread(r))) for r in df.ruta]); y = df.clase.values
# print(X.shape)

## 5. Modelo y evaluación

In [ ]:
# TODO: X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=SEMILLA)
modelos = {
    "k-NN": Pipeline([("escala", StandardScaler()), ("clf", KNeighborsClassifier(n_neighbors=5))]),
    "SVM": Pipeline([("escala", StandardScaler()), ("clf", SVC())]),
    # TODO: agrega otro modelo (árbol o RandomForestClassifier)
}
# TODO: para cada modelo, cross_val_score(modelo, X_tr, y_tr, cv=StratifiedKFold(5, shuffle=True, random_state=SEMILLA), scoring="f1_macro")
# TODO: entrena el mejor modelo, evalúa en prueba: classification_report y ConfusionMatrixDisplay.from_estimator
# Alternativa no supervisada: KMeans(n_clusters=número de clases) y tabla de contingencia grupos vs. etiquetas

### Ejemplos mal clasificados

In [ ]:
# TODO: muestra algunas imágenes de prueba mal clasificadas con su etiqueta real y la predicha

## 6. Análisis y discusión
- ¿Qué tan bien funcionó el flujo propuesto?
- ¿Cuáles fueron las principales fuentes de error o dificultades?
- ¿Qué cambios harías en una versión mejorada (más datos, otros descriptores, otro modelo)?

## Referencias y uso de herramientas
*Lista en formato APA las fuentes consultadas. Si usaste asistentes de IA generativa, indica cuál, para qué y verifica que entiendes cada línea de código que entregas.*

---
**Antes de entregar:** `Entorno de ejecución → Reiniciar y ejecutar todo`, verifica que no haya errores y exporta a PDF (`Archivo → Imprimir → Guardar como PDF` o `jupyter nbconvert --to webpdf`).